# Clase 16 — Type Hints y el módulo `typing`

Los **type hints** (anotaciones de tipo) son una de las adiciones más importantes de Python moderno. Te permiten documentar qué tipos espera tu código, detectar errores antes de ejecutarlo, y mejorar el autocompletado en tu editor.

**Contenido de esta clase:**
1. ¿Qué son los type hints?
2. Sintaxis básica
3. Tipos compuestos (List, Dict, Tuple, Set)
4. Tipos opcionales y unión (Optional, Union)
5. Anotaciones en funciones
6. Callable, Iterator, Generator
7. Any, NoReturn, Never
8. TypedDict y Protocol
9. Alias y NewType
10. `mypy` y verificación estática
11. Ventajas y desventajas
12. Ejercicio integrador


## 1. ¿Qué son los type hints?

Los **type hints** son anotaciones opcionales que indican el **tipo esperado** de variables, parámetros y valores de retorno. Fueron introducidos en **PEP 484** (Python 3.5+) y mejorados en versiones posteriores.

### ¿Para qué sirven?

1. **Documentación viva:** el tipo es parte del código, no solo del docstring.
2. **Detección temprana de errores:** herramientas como `mypy` y `pyright` verifican tipos estáticamente.
3. **Mejor autocompletado:** los editores (VS Code, PyCharm) saben qué métodos están disponibles.
4. **Refactorización segura:** cambiar tipos se detecta en tiempo de compilación, no en producción.

### ¿Son obligatorios?

**No.** Python sigue siendo dinámico. Los type hints son **opcionales** y se ignoran en tiempo de ejecución. Tu código sigue funcionando igual sin ellos.


In [ ]:
# Sin type hints
def saludar(nombre):
    return f"Hola, {nombre}!"

# Con type hints
def saludar(nombre: str) -> str:
    return f"Hola, {nombre}!"

# Python ignora los hints en tiempo de ejecución
print(saludar("Ana"))  # Funciona igual
print(saludar(42))    # También funciona, aunque no tiene sentido


## 2. Sintaxis básica

La sintaxis es simple: `variable: tipo` para variables, `parametro: tipo` para parámetros, y `-> tipo` para el valor de retorno.


In [1]:
# Variables básicas
edad: int = 25
nombre: str = "Ana"
promedio: float = 8.5
es_aprobado: bool = True
iniciales: bytes = b"AB"

print(f"{nombre}, {edad} años, promedio {promedio}, aprobado: {es_aprobado}")


Ana, 25 años, promedio 8.5, aprobado: True


In [2]:
# El tipo NO se verifica en tiempo de ejecución
x: int = "esto es un string"  # No lanza error en ejecución
print(x)  # Funciona

# Para verificar, necesitas mypy (lo veremos al final)


esto es un string


## 3. Tipos compuestos

Para colecciones, usa los tipos del módulo `typing` (Python < 3.9) o los built-ins (Python ≥ 3.9).


In [3]:
from typing import List, Dict, Tuple, Set

# Python ≥ 3.9: puedes usar los built-ins directamente
# Python < 3.9: necesitas importar List, Dict, Tuple, Set desde typing

# Listas
numeros: List[int] = [1, 2, 3, 4, 5]
nombres: list[str] = ["Ana", "Luis", "María"]  # Python 3.9+

# Diccionarios
edades: Dict[str, int] = {"Ana": 25, "Luis": 30}
calificaciones: dict[str, float] = {"matemáticas": 9.5, "física": 8.0}  # 3.9+

# Tuplas
coordenadas: Tuple[float, float] = (3.14, 2.71)
colores_rgb: tuple[int, int, int] = (255, 0, 0)  # 3.9+

# Sets
vocales: Set[str] = {"a", "e", "i", "o", "u"}
numeros_unicos: set[int] = {1, 2, 3, 4, 5}  # 3.9+

print(f"numeros: {numeros}")
print(f"coordenadas: {coordenadas}")
print(f"vocales: {vocales}")


numeros: [1, 2, 3, 4, 5]
coordenadas: (3.14, 2.71)
vocales: {'i', 'u', 'a', 'e', 'o'}


## 4. Tipos opcionales y unión

`Optional[X]` significa "X o None". `Union[X, Y]` significa "X o Y". En Python 3.10+ puedes usar `X | Y` y `X | None`.


In [4]:
from typing import Optional, Union

# Optional[X] es equivalente a Union[X, None]
def buscar_estudiante(matricula: str) -> Optional[dict]:
    """Busca un estudiante. Retorna None si no existe."""
    estudiantes = {"A001": {"nombre": "Ana", "edad": 25}}
    return estudiantes.get(matricula)

resultado = buscar_estudiante("A001")
print(f"Resultado: {resultado}")

resultado = buscar_estudiante("B999")
print(f"Resultado: {resultado}")


Resultado: {'nombre': 'Ana', 'edad': 25}
Resultado: None


In [5]:
# Python 3.10+: sintaxis moderna con |
def dividir(a: float, b: float) -> float | None:
    """Divide a entre b. Retorna None si b es 0."""
    if b == 0:
        return None
    return a / b

print(f"10 / 2 = {dividir(10, 2)}")
print(f"10 / 0 = {dividir(10, 0)}")


10 / 2 = 5.0
10 / 0 = None


In [6]:
# Union: múltiples tipos posibles
from typing import Union

def procesar_id(id: Union[int, str]) -> str:
    """Acepta un ID como int o str y lo retorna como string."""
    return str(id)

print(procesar_id(123))
print(procesar_id("ABC-456"))

# Python 3.10+: con |
def procesar_id_moderno(id: int | str) -> str:
    return str(id)

print(procesar_id_moderno(789))


123
ABC-456
789


## 5. Anotaciones en funciones

Los type hints brillan en las funciones: documentan contratos claros.


In [7]:
# Función básica con hints
def calcular_promedio(notas: list[float]) -> float:
    """Calcula el promedio de una lista de notas."""
    if not notas:
        return 0.0
    return sum(notas) / len(notas)

# Función con múltiples parámetros y retorno opcional
def buscar_calificacion(
    calificaciones: dict[str, float],
    materia: str,
) -> float | None:
    """Busca la calificación de una materia. Retorna None si no existe."""
    return calificaciones.get(materia)

# Función que retorna múltiples valores
def estadisticas(notas: list[float]) -> tuple[float, float, float]:
    """Retorna (mínimo, máximo, promedio)."""
    return min(notas), max(notas), sum(notas) / len(notas)

notas = [8.5, 9.0, 7.5, 10.0]
print(f"Promedio: {calcular_promedio(notas)}")
print(f"Estadísticas: {estadisticas(notas)}")


Promedio: 8.75
Estadísticas: (7.5, 10.0, 8.75)


## 6. Callable, Iterator, Generator

`Callable` representa funciones. `Iterator` y `Generator` son para objetos iterables.


In [8]:
from typing import Callable

# Función que recibe otra función como argumento
def aplicar_dos_veces(func: Callable[[int], int], valor: int) -> int:
    """Aplica func dos veces al valor."""
    return func(func(valor))

def doble(x: int) -> int:
    return x * 2

print(aplicar_dos_veces(doble, 5))  # doble(doble(5)) = 20


20


In [9]:
from typing import Iterator, Generator

# Iterator: cualquier objeto sobre el que podamos iterar
def primeros_n(n: int) -> Iterator[int]:
    """Genera los primeros n números naturales."""
    i = 0
    while i < n:
        yield i
        i += 1

# Generator: tipo más específico que Iterator
def contador_hasta(maximo: int) -> Generator[int, None, None]:
    """Genera números del 0 al máximo."""
    i = 0
    while i <= maximo:
        yield i
        i += 1

for num in primeros_n(3):
    print(f"Iterator: {num}")

print("---")

for num in contador_hasta(3):
    print(f"Generator: {num}")


Iterator: 0
Iterator: 1
Iterator: 2
---
Generator: 0
Generator: 1
Generator: 2
Generator: 3


## 7. Any, NoReturn, Never

`Any` desactiva la verificación de tipo. `NoReturn` y `Never` indican que una función nunca retorna normalmente.


In [10]:
from typing import Any, NoReturn, Never

# Any: cualquier tipo (escape hatch)
def procesar_dato(dato: Any) -> Any:
    """Procesa un dato de cualquier tipo. Usar con cuidado."""
    return dato

print(procesar_dato(42))
print(procesar_dato("hola"))
print(procesar_dato([1, 2, 3]))


42
hola
[1, 2, 3]


In [11]:
import sys

# NoReturn: función que siempre lanza excepción
def error_fatal(mensaje: str) -> NoReturn:
    """Imprime un mensaje y termina el programa."""
    print(f"ERROR FATAL: {mensaje}", file=sys.stderr)
    sys.exit(1)

# Never: tipo más moderno (Python 3.11+)
def loop_infinito() -> Never:
    """Loop que nunca termina."""
    while True:
        pass

# Si llamas a error_fatal, el programa termina
# error_fatal("Algo salió muy mal")
print("El programa continúa (no llamamos a error_fatal)")


El programa continúa (no llamamos a error_fatal)


## 8. TypedDict y Protocol

`TypedDict` define diccionarios con claves y valores tipados. `Protocol` permite duck typing estático.


In [12]:
from typing import TypedDict

# TypedDict: diccionario con schema conocido
class Estudiante(TypedDict):
    nombre: str
    edad: int
    promedio: float
    activo: bool

# Crear instancias (son solo diccionarios con tipos)
ana: Estudiante = {
    "nombre": "Ana",
    "edad": 25,
    "promedio": 9.5,
    "activo": True
}
print(f"ana: {ana}")

# mypy verificará que todas las claves estén presentes y con tipo correcto


ana: {'nombre': 'Ana', 'edad': 25, 'promedio': 9.5, 'activo': True}


In [13]:
from typing import Protocol

# Protocol: duck typing estático
class Hablador(Protocol):
    def hablar(self) -> str: ...

class Perro:
    def hablar(self) -> str:
        return "Guau!"

class Gato:
    def hablar(self) -> str:
        return "Miau!"

def hacer_hablar(animal: Hablador) -> str:
    return animal.hablar()

# mypy acepta cualquier objeto con método hablar()
print(hacer_hablar(Perro()))
print(hacer_hablar(Gato()))


Guau!
Miau!


## 9. Alias y NewType

`TypeAlias` crea nombres alternativos. `NewType` crea tipos distintos para validación.


In [14]:
from typing import TypeAlias, NewType

# TypeAlias: nombre alternativo para un tipo
Vector: TypeAlias = list[float]
Matriz: TypeAlias = list[list[float]]

def sumar_vectores(a: Vector, b: Vector) -> Vector:
    return [x + y for x, y in zip(a, b)]

v1: Vector = [1.0, 2.0, 3.0]
v2: Vector = [4.0, 5.0, 6.0]
print(f"v1 + v2 = {sumar_vectores(v1, v2)}")


v1 + v2 = [5.0, 7.0, 9.0]


In [15]:
# NewType: crea un tipo "fuerte" (stricter que alias)
UsuarioId = NewType("UsuarioId", int)
NombreUsuario = NewType("NombreUsuario", str)

def obtener_usuario(user_id: UsuarioId) -> NombreUsuario:
    """Busca un usuario por ID y retorna su nombre."""
    return NombreUsuario(f"Usuario_{user_id}")

# mypy verificará que pases un int como UsuarioId
uid = UsuarioId(42)
nombre = obtener_usuario(uid)
print(f"Usuario {uid}: {nombre}")


Usuario 42: Usuario_42


## 10. `mypy` y verificación estática

`mypy` es la herramienta oficial para verificar tipos estáticamente.

### Instalación

```bash
uv add --dev mypy
```

### Uso

```bash
uv run mypy mi_archivo.py
```

`mypy` lee tus type hints y reporta errores **antes de ejecutar** el código.


In [16]:
# Ejemplo de código que mypy detecta como error
# (Este código FUNCIONA en Python, pero mypy lo rechaza)

from typing import List

def suma(numeros: List[int]) -> int:
    return sum(numeros)

# mypy detecta que "1" + 2 son tipos incompatibles
# resultado = suma("123")  # Error en mypy, pero no en Python

# El código correcto:
resultado = suma([1, 2, 3])
print(f"Suma: {resultado}")


Suma: 6


## 11. Ventajas y desventajas de los type hints

### Ventajas

| Ventaja | Descripción |
|---|---|
| **Documentación viva** | El tipo es parte del código, siempre actualizado. |
| **Detección temprana** | `mypy`/`pyright` encuentran errores antes de ejecutar. |
| **Mejor autocompletado** | Los editores saben qué métodos están disponibles. |
| **Refactorización segura** | Cambiar tipos se valida estáticamente. |
| **Documentación para equipos** | Nuevos desarrolladores entienden las APIs más rápido. |
| **Contratos claros** | Las funciones declaran qué aceptan y qué retornan. |

### Desventajas

| Desventaja | Descripción |
|---|---|
| **Más código para escribir** | Hay que anotar cada variable y función. |
| **Verbosidad** | `List[Dict[str, int]]` vs `[{str: int}]` (en Python 3.9+). |
| **Curva de aprendizaje** | Hay que aprender el módulo `typing`. |
| **No se verifica en runtime** | Sin `mypy`, los errores pasan hasta producción. |
| **Falsa seguridad** | `Any` desactiva la verificación; hay que usarlo con cuidado. |
| **Overhead inicial** | En proyectos pequeños puede ser excesivo. |

### ¿Cuándo usar type hints?

| Situación | ¿Usar? |
|---|---|
| Librería o API pública | ✅ Sí, muy recomendado |
| Proyecto de equipo mediano/grande | ✅ Sí |
| Script personal pequeño | ❌ Opcional |
| Prototipo rápido | ❌ Añádelo después si el prototipo crece |
| Jupyter Notebook exploratorio | ❌ No vale la pena |


## 12. Ejercicio Práctico: Sistema de calificaciones con type hints

Vas a crear un módulo de gestión de calificaciones usando type hints completos, documentación PEP 257, y principios del Zen.

**Instrucciones:**

Crea un archivo `calificaciones_typed.py` con:

1. `TypedDict Estudiante` con: `nombre: str`, `matricula: str`, `calificaciones: list[float]`.
2. Función `agregar_calificacion(estudiante: Estudiante, nota: float) -> Estudiante`.
3. Función `calcular_promedio(estudiante: Estudiante) -> float | None` (None si no hay calificaciones).
4. Función `aprobo(estudiante: Estudiante, minimo: float = 7.0) -> bool`.
5. Función `ordenar_por_promedio(estudiantes: list[Estudiante]) -> list[Estudiante]`.
6. Usa `NewType` para `Matricula` (distinta de un `int` cualquiera).
7. Todas las funciones con docstrings estilo Google.
8. Sigue PEP 8.

**Pistas:**
- Usa `Estudiante` como `TypedDict` (no clase) para mantenerlo simple.
- `NewType("Matricula", str)` crea un tipo "fuerte" para matrículas.
- `from typing import TypedDict, NewType`.

**Restricción del ejercicio:**
- No uses `Any` (degrada la verificación).
- Todas las funciones deben tener type hints completos.
- Sigue PEP 8 y Zen de Python.


In [17]:
"""
Solución al Ejercicio Práctico
Este código debe ir en un archivo calificaciones_typed.py
"""

from typing import TypedDict, NewType

# Tipo fuerte para matrículas (distinto de cualquier str)
Matricula = NewType("Matricula", str)


class Estudiante(TypedDict):
    """Representa un estudiante con sus calificaciones."""
    nombre: str
    matricula: Matricula
    calificaciones: list[float]


def agregar_calificacion(
    estudiante: Estudiante, nota: float
) -> Estudiante:
    """Agrega una calificación al estudiante.
    
    Args:
        estudiante: Diccionario con los datos del estudiante.
        nota: Calificación a agregar (debe estar entre 0 y 10).
    
    Returns:
        El mismo diccionario del estudiante, ahora con la nueva nota.
    
    Raises:
        ValueError: Si la nota no está entre 0 y 10.
    """
    if not 0 <= nota <= 10:
        raise ValueError(f"Nota inválida: {nota}. Debe estar entre 0 y 10.")
    estudiante["calificaciones"].append(nota)
    return estudiante


def calcular_promedio(estudiante: Estudiante) -> float | None:
    """Calcula el promedio de las calificaciones del estudiante.
    
    Args:
        estudiante: Diccionario con los datos del estudiante.
    
    Returns:
        El promedio redondeado a 2 decimales, o None si no hay calificaciones.
    """
    notas = estudiante["calificaciones"]
    if not notas:
        return None
    return round(sum(notas) / len(notas), 2)


def aprobo(estudiante: Estudiante, minimo: float = 7.0) -> bool:
    """Determina si un estudiante aprobó según su promedio.
    
    Args:
        estudiante: Diccionario con los datos del estudiante.
        minimo: Calificación mínima para aprobar (default 7.0).
    
    Returns:
        True si el promedio es >= minimo, False en caso contrario.
    """
    promedio = calcular_promedio(estudiante)
    if promedio is None:
        return False
    return promedio >= minimo


def ordenar_por_promedio(
    estudiantes: list[Estudiante],
) -> list[Estudiante]:
    """Ordena una lista de estudiantes por promedio, de mayor a menor.
    
    Args:
        estudiantes: Lista de diccionarios de estudiantes.
    
    Returns:
        La misma lista ordenada de mayor a menor promedio.
        Estudiantes sin calificaciones van al final.
    """
    def clave_orden(est: Estudiante) -> float:
        promedio = calcular_promedio(est)
        return promedio if promedio is not None else -1.0
    
    return sorted(estudiantes, key=clave_orden, reverse=True)


In [18]:
# Probar el sistema completo
ana: Estudiante = {
    "nombre": "Ana",
    "matricula": Matricula("A001"),
    "calificaciones": [9.0, 8.5, 9.5, 10.0]
}
luis: Estudiante = {
    "nombre": "Luis",
    "matricula": Matricula("B002"),
    "calificaciones": [6.0, 5.5, 7.0]
}
maria: Estudiante = {
    "nombre": "María",
    "matricula": Matricula("C003"),
    "calificaciones": []
}

# Agregar calificaciones
agregar_calificacion(ana, 9.5)
agregar_calificacion(luis, 6.5)

# Calcular promedios
for est in [ana, luis, maria]:
    promedio = calcular_promedio(est)
    estado = "Aprobado" if aprobo(est) else "Reprobado"
    print(f"{est['nombre']}: promedio={promedio}, estado={estado}")

print("\n--- Ranking ---")
for est in ordenar_por_promedio([luis, maria, ana]):
    promedio = calcular_promedio(est)
    print(f"  {est['nombre']}: {promedio}")


Ana: promedio=9.3, estado=Aprobado
Luis: promedio=6.25, estado=Reprobado
María: promedio=None, estado=Reprobado

--- Ranking ---
  Ana: 9.3
  Luis: 6.25
  María: None


## Resumen de la clase

En esta clase aprendimos:

1. **Type hints:** anotaciones opcionales que documentan tipos esperados.
2. **Sintaxis básica:** `: tipo` para variables y parámetros, `-> tipo` para retornos.
3. **Tipos compuestos:** `List[T]`, `Dict[K, V]`, `Tuple`, `Set` (o built-ins en 3.9+).
4. **Optional y Union:** `Optional[X]` o `X | None`, `Union[X, Y]` o `X | Y`.
5. **Callable, Iterator, Generator:** tipos para funciones y objetos iterables.
6. **Any, NoReturn, Never:** tipos especiales para casos específicos.
7. **TypedDict y Protocol:** diccionarios tipados y duck typing estático.
8. **Alias y NewType:** nombres alternativos y tipos "fuertes".
9. **`mypy`:** verificación estática de tipos.
10. **Cuándo usar type hints:** en APIs públicas, equipos grandes, y cuando el código crece.

**En la siguiente clase** (Clase 17) veremos **Programación Funcional**: funciones puras, `map`/`filter`/`reduce`, comprehensions y `functools`.


## Recursos adicionales

- **PEP 484 (Type Hints):** https://peps.python.org/pep-0484/
- **PEP 585 (Genéricos built-in):** https://peps.python.org/pep-0585/ (Python 3.9+)
- **Documentación de `typing`:** https://docs.python.org/3/library/typing.html
- **`mypy`:** http://mypy-lang.org/
- **`pyright` (alternativa de Microsoft):** https://github.com/microsoft/pyright
